<a href="https://colab.research.google.com/github/giaaakk/Komputasi-Numerik/blob/main/SPL_Gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**# PROGRAM ELIMINASI GAUS**

In [ ]:
import numpy as np

EPS = 1e-9  # toleransi untuk dianggap nol

numpy dipakai untuk operasi matriks

*   numpy dipakai untuk operasi matriks (array, perkalian, dsb.).
*   EPS adalah toleransi. Komputer menyimpan bilangan desimal dengan sedikit pembulatan, sehingga hasil hitungan yang seharusnya 0 bisa muncul sebagai 0.0000000001. Dengan EPS, nilai yang lebih kecil dari 1e-9 dianggap nol.

# Penjelasan Umum Program Eliminasi Gauss

1. Masalah yang diselesaikan

Sistem persamaan linear (SPL) adalah kumpulan persamaan dengan beberapa variabel, misalnya:

 2x +  y -  z =   8
-3x -  y + 2z = -11
-2x +  y + 2z =  -3

Tujuannya mencari nilai x, y, z yang memenuhi semua persamaan sekaligus. Metode Gauss melakukannya dengan menyederhanakan sistem sampai bisa diselesaikan satu per satu.





In [ ]:
def tampilkan_matriks(M, judul=""):
    if judul:
        print(judul)
    n, m = M.shape
    for i in range(n):
        baris = "  ".join(f"{M[i, j]:9.4f}" for j in range(m - 1))
        print(f"[ {baris}  |  {M[i, -1]:9.4f} ]")
    print()

*   Fungsi ini hanya mencetak matriks augmented dengan rapi agar langkah eliminasi mudah dibaca.
*   M.shape mengambil jumlah baris (n) dan kolom (m).
*   Kolom terakhir adalah konstanta b, jadi dicetak terpisah setelah tanda |.
*   {...:9.4f} berarti angka dicetak selebar 9 karakter dengan 4 angka di belakang koma.
*   Parameter judul bersifat opsional, untuk keterangan seperti “Tukar baris R1 <-> R2”.

# Penjelasan Umum Program Eliminasi Gauss

2. Ide dasar metode Gauss

Metode ini terdiri dari dua tahap:

Eliminasi maju: membuat angka di bawah diagonal menjadi 0, sehingga matriks berbentuk segitiga atas. Persamaan paling bawah akhirnya hanya berisi satu variabel.
Substitusi mundur: variabel terakhir dihitung lebih dulu, lalu nilainya dimasukkan ke persamaan di atasnya, dan seterusnya sampai variabel pertama.

Seluruh sistem ditulis sebagai matriks augmented [A | b], yaitu koefisien dan konstanta digabung dalam satu matriks.

In [ ]:
def eliminasi_maju(A, b, tampilkan_langkah=True):
    """
    Mengubah matriks augmented [A|b] menjadi bentuk eselon baris
    dengan partial pivoting.
    Return: matriks eselon, daftar kolom pivot
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    M = np.hstack([A, b])
    n, m = A.shape

    if tampilkan_langkah:
        tampilkan_matriks(M, "Matriks augmented awal:")

    baris = 0
    kolom_pivot = []

    for kol in range(m):
        if baris >= n:
            break

        # Cari pivot terbesar (partial pivoting)
        p = baris + np.argmax(np.abs(M[baris:, kol]))
        if abs(M[p, kol]) < EPS:
            continue  # kolom ini tidak punya pivot -> variabel bebas

        # Tukar baris bila perlu
        if p != baris:
            M[[baris, p]] = M[[p, baris]]
            if tampilkan_langkah:
                tampilkan_matriks(M, f"Tukar baris R{baris+1} <-> R{p+1}:")

        # Eliminasi elemen di bawah pivot
        for i in range(baris + 1, n):
            faktor = M[i, kol] / M[baris, kol]
            if abs(faktor) < EPS:
                continue
            M[i] = M[i] - faktor * M[baris]
            M[i, kol] = 0.0
            if tampilkan_langkah:
                tampilkan_matriks(
                    M, f"R{i+1} = R{i+1} - ({faktor:.4f}) * R{baris+1}:"
                )

        kolom_pivot.append(kol)
        baris += 1

    return M, kolom_pivot

ini inti metode Gauss, yaitu mengubah matriks menjadi bentuk eselon baris (segitiga atas).

1. Membentuk matriks augmented: np.hstack([A, b]) menempelkan vektor b di sisi kanan A.
2. Variabel pelacak: baris adalah baris pivot yang sedang diproses, kolom_pivot mencatat kolom yang punya pivot (berguna untuk menentukan variabel bebas).
3. Loop per kolom: untuk setiap kolom, program mencari pivot.
4. Partial pivoting: np.argmax(np.abs(M[baris:, kol])) memilih elemen bernilai mutlak terbesar di kolom itu. Ini mengurangi kesalahan pembulatan dan mencegah pembagian dengan angka sangat kecil.
5. Kolom tanpa pivot: jika semua elemennya nol (< EPS), kolom dilewati dengan continue. Variabel di kolom ini nantinya menjadi variabel bebas.
6. Tukar baris: jika pivot tidak berada di posisi baris, kedua baris ditukar.
7. Eliminasi: untuk tiap baris di bawah pivot, dihitung faktor = elemen / pivot, lalu baris itu dikurangi faktor × baris pivot, sehingga elemen di bawah pivot menjadi 0.
8. Catat pivot lalu pindah ke baris berikutnya (baris += 1).
9. Fungsi mengembalikan matriks eselon dan daftar kolom pivot.

# Penjelasan Umum Program Eliminasi Gauss

3. Alur kerja program

Input A dan b
      ↓
Bentuk matriks augmented [A|b]
      ↓
Untuk tiap kolom:
   cari pivot terbesar → tukar baris → nolkan elemen di bawah pivot
      ↓
Periksa hasil eliminasi
   ├─ ada baris [0 0 0 | c], c ≠ 0  → tidak ada solusi
   ├─ pivot < jumlah variabel        → solusi tak hingga
   └─ pivot = jumlah variabel        → solusi tunggal → substitusi mundur

Alur ini sama dengan yang diminta di papan: nanti bisa dijadikan dasar untuk membuat deskripsi algoritma, pseudo code, dan flow chart.



In [ ]:
def selesaikan_spl(A, b, tampilkan_langkah=True):
    M, kolom_pivot = eliminasi_maju(A, b, tampilkan_langkah)
    n, m = M.shape[0], M.shape[1] - 1
    rank = len(kolom_pivot)

    # Cek tidak konsisten: baris [0 0 ... 0 | c] dengan c != 0
    for i in range(n):
        if np.all(np.abs(M[i, :m]) < EPS) and abs(M[i, m]) > EPS:
            print("Hasil: SPL TIDAK MEMILIKI SOLUSI.")
            return None

    # Solusi tak hingga
    if rank < m:
        bebas = [j + 1 for j in range(m) if j not in kolom_pivot]
        print("Hasil: SPL MEMILIKI SOLUSI TAK HINGGA.")
        print(f"Variabel bebas: {', '.join('x' + str(j) for j in bebas)}")
        return None

    # Solusi tunggal -> substitusi mundur
    x = np.zeros(m)
    for r in range(rank - 1, -1, -1):
        c = kolom_pivot[r]
        x[c] = (M[r, -1] - np.dot(M[r, c + 1:m], x[c + 1:m])) / M[r, c]

    print("Hasil: SPL MEMILIKI SOLUSI TUNGGAL.")
    for i, nilai in enumerate(x, start=1):
        print(f"x{i} = {nilai:.4f}")
    return x

fungsi ini memanggil eliminasi maju, lalu menentukan jenis solusi dan menghitung nilainya.

* rank = len(kolom_pivot): jumlah pivot, yaitu rank matriks.
* Cek tidak ada solusi: jika ada baris berbentuk [0 0 ... 0 | c] dengan c ≠ 0, persamaannya berbunyi 0 = c, yang mustahil. Program mencetak bahwa SPL tidak memiliki solusi.
* Cek solusi tak hingga: jika rank < m (pivot kurang dari jumlah variabel), ada variabel bebas. Program menampilkan variabel mana yang bebas (kolom yang tidak ada di kolom_pivot).
* Solusi tunggal (substitusi mundur): jika rank sama dengan jumlah variabel, nilai variabel dihitung dari baris paling bawah ke atas dengan rumus: x[c] = (konstanta − jumlah koefisien × variabel yang sudah diketahui) / pivot Bagian np.dot(M[r, c+1:m], x[c+1:m]) adalah jumlah koefisien dikali nilai variabel yang sudah ditemukan.

# Penjelasan Umum Program Eliminasi Gauss

4. Simulasi dengan contoh soal

Matriks awal:

[  2   1  -1 |   8 ]
[ -3  -1   2 | -11 ]
[ -2   1   2 |  -3 ]

Kolom 1: elemen terbesar (mutlak) adalah -3 di baris 2, jadi R1 ditukar dengan R2.

[ -3  -1   2 | -11 ]
[  2   1  -1 |   8 ]
[ -2   1   2 |  -3 ]

Lalu R2 dan R3 dikurangi kelipatan R1 agar kolom 1 di bawah pivot menjadi 0:

[ -3  -1      2    | -11     ]
[  0   0.3333 0.3333|  0.6667 ]
[  0   1.6667 0.6667|  4.3333 ]

Kolom 2: elemen terbesar adalah 1.6667 (baris 3), jadi R2 ditukar dengan R3, lalu R3 dikurangi 0.2 × R2:

[ -3  -1      2      | -11     ]
[  0   1.6667 0.6667 |  4.3333 ]
[  0   0      0.2    | -0.2    ]

Matriks sudah berbentuk segitiga atas.

Substitusi mundur:

Dari baris 3: 0.2·x3 = -0.2, jadi x3 = -1
Dari baris 2: 1.6667·x2 + 0.6667·(-1) = 4.3333, jadi x2 = 3
Dari baris 1: -3·x1 - 3 + 2(-1) = -11, jadi x1 = 2

Hasil ini sama dengan output program.

In [ ]:
# Contoh:
#  2x +  y -  z =   8
# -3x -  y + 2z = -11
# -2x +  y + 2z =  -3
A = [[ 2,  1, -1],
     [-3, -1,  2],
     [-2,  1,  2]]
b = [8, -11, -3]

hasil = selesaikan_spl(A, b)

* A adalah matriks koefisien dan b adalah vektor konstanta dari sistem persamaan di komentar.
* selesaikan_spl(A, b) dijalankan dan seluruh langkah eliminasi ikut tercetak.
* Hasilnya disimpan di hasil agar bisa dipakai di sel berikutnya.
* Solusi yang diharapkan: x1 = 2, x2 = 3, x3 = -1.

# Penjelasan Umum Program Eliminasi Gauss

5. Konsep penting dalam code

**Pivot** adalah elemen diagonal yang dipakai untuk menghilangkan elemen-elemen di bawahnya dalam satu kolom. Pada setiap kolom, program memilih satu elemen sebagai pivot, lalu menggunakannya sebagai acuan untuk membuat angka-angka di bawahnya menjadi nol melalui operasi baris.

**Partial pivoting** adalah teknik memilih pivot dengan nilai mutlak terbesar di kolom yang sedang diproses, lalu menukar barisnya ke posisi pivot. Tujuannya agar hasil perhitungan lebih akurat dan program terhindar dari pembagian dengan angka yang nol atau hampir nol, yang bisa menyebabkan error atau kesalahan pembulatan yang besar.

**EPS (toleransi)** adalah angka yang sangat kecil, yaitu 1e-9, yang dipakai sebagai batas untuk menganggap suatu nilai sama dengan nol. Hal ini diperlukan karena komputer menyimpan bilangan desimal dengan pembulatan, sehingga hasil hitungan yang seharusnya tepat nol bisa muncul sebagai angka seperti 0.0000000001. Tanpa toleransi ini, program bisa salah mengira angka tersebut sebagai pivot yang valid.

**Rank** adalah jumlah pivot yang ditemukan selama eliminasi maju. Program membandingkan rank dengan jumlah variabel untuk menentukan jenis solusi: jika rank sama dengan jumlah variabel, solusinya tunggal, sedangkan jika rank lebih kecil, ada variabel yang tidak terikat sehingga solusinya tak hingga (selama tidak ada kontradiksi).

**Variabel bebas** adalah variabel yang berada pada kolom tanpa pivot. Karena tidak ada persamaan yang mengikat nilainya secara khusus, variabel ini boleh bernilai apa saja, dan nilai variabel lainnya menyesuaikan. Itulah sebabnya keberadaan variabel bebas menghasilkan solusi yang jumlahnya tak hingga.

In [ ]:
if hasil is not None:
    print("A @ x =", np.round(np.array(A) @ hasil, 4))
    print("b     =", np.array(b))
    print("Cocok dengan numpy.linalg.solve?",
          np.allclose(hasil, np.linalg.solve(A, b)))

* np.array(A) @ hasil mengalikan matriks A dengan solusi x. Hasilnya harus sama dengan b.
* np.linalg.solve(A, b) adalah fungsi bawaan NumPy untuk menyelesaikan SPL. np.allclose membandingkan hasilnya dengan hasil buatan kita. Jika True, program Gauss sudah benar.
* Pengecekan if hasil is not None mencegah error ketika SPL tidak punya solusi tunggal.

# Penjelasan Umum Program Eliminasi Gauss

6. Tiga kemungkinan hasil SPL

Solusi tunggal terjadi ketika sistem konsisten dan jumlah pivot sama dengan jumlah variabel. Setelah eliminasi maju, setiap variabel punya satu persamaan yang menentukannya, sehingga baris terakhir berbentuk seperti [0 0 0.2 | -0.2]. Dari baris ini nilai variabel terakhir langsung diperoleh, lalu substitusi mundur menghitung variabel lainnya. Hasilnya hanya satu kombinasi nilai yang memenuhi semua persamaan.

Tidak ada solusi terjadi ketika setelah eliminasi muncul baris yang semua koefisiennya nol tetapi konstantanya tidak nol, misalnya [0 0 0 | 5]. Baris ini berarti persamaan 0 = 5, yang mustahil terpenuhi berapa pun nilai variabelnya. Karena ada satu persamaan yang bertentangan, seluruh sistem tidak punya solusi, dan program langsung mencetak bahwa SPL tidak memiliki solusi.

Solusi tak hingga terjadi ketika tidak ada kontradiksi, tetapi jumlah pivot lebih sedikit daripada jumlah variabel. Biasanya ini terlihat dari adanya baris [0 0 0 | 0], yang berarti persamaan tersebut tidak memberi informasi baru karena hanya mengulang persamaan lain. Akibatnya ada variabel bebas yang nilainya boleh dipilih sembarang, dan nilai variabel lainnya menyesuaikan, sehingga jumlah solusinya tidak terbatas.


In [ ]:
n = int(input("Jumlah persamaan: "))
m = int(input("Jumlah variabel : "))

A_in, b_in = [], []
for i in range(n):
    baris = list(map(float, input(f"Koefisien persamaan {i+1} (pisahkan spasi): ").split()))
    assert len(baris) == m, "Jumlah koefisien harus sama dengan jumlah variabel"
    A_in.append(baris)
    b_in.append(float(input(f"Konstanta persamaan {i+1}: ")))

print()
selesaikan_spl(A_in, b_in)

* input() membaca jumlah persamaan (n) dan jumlah variabel (m).
* Untuk tiap persamaan, pengguna mengetik koefisien dipisah spasi, misalnya 2 1 -1, lalu konstantanya.
* map(float, ...split()) mengubah teks menjadi bilangan desimal.
* assert memastikan jumlah koefisien sesuai jumlah variabel. Jika tidak, program berhenti dengan pesan kesalahan.
* Data kemudian dikirim ke selesaikan_spl.

# Penjelasan Umum Program Eliminasi Gauss

7. Struktur program

Program dibagi menjadi fungsi-fungsi kecil agar rapi dan mudah diuji:

* tampilkan_matriks: bertugas menampilkan.
* eliminasi_maju: bertugas menghitung (tahap 1).
* selesaikan_spl: mengatur alur, menentukan jenis solusi, dan melakukan substitusi mundur (tahap 2).

Pemisahan ini membuat tiap fungsi punya satu tugas dan mudah diperbaiki bila ada kesalahan.

8. Kelebihan dan keterbatasan

Kelebihan:

* Menangani ketiga jenis solusi.
* Menampilkan langkah-langkah, jadi cocok untuk belajar dan laporan.
* Memakai partial pivoting sehingga lebih stabil.

Keterbatasan:

* Untuk solusi tak hingga, program hanya menyebutkan variabel bebasnya, belum menuliskan bentuk parametriknya.
* Untuk matriks yang sangat besar atau kondisinya buruk, hasil bisa kurang akurat karena pembulatan.